In [1]:
!pip install yfinance

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


def data(Ticker  , start , end) :
  x = yf.download(Ticker  , start , end , multi_level_index=False)
  x = x.ffill()
  return x

df1 = data('HDFCBANK.NS' , '2020-01-01' , date.today())
df2 = data('TCS.NS' , '2020-01-01' , date.today())
df3 = data('RELIANCE.NS' , '2020-01-01' , date.today())
df4 = data('SUNPHARMA.NS' , '2020-01-01' , date.today())
df5 = data('HINDUNILVR.NS' , '2020-01-01' , date.today())
df6 = data('BHARTIARTL.NS' , '2020-01-01' , date.today())
df7 = data('M&M.NS' , '2020-01-01' , date.today())
df8 = data('LT.NS' , '2020-01-01' , date.today())
df_b = data('^NSEI' , '2020-01-01' , date.today())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [7]:
#backtest

def backtest(signal , close, fee=0.0025 , alloc = 0.25 , capital = 1000000) :
  shares = ( (capital * alloc) / close.shift(1) ).fillna(0).astype(int)
  daily_pnl = close.diff() * signal.shift(1)
  position = signal.diff().fillna(0)
  daily_fees = fee * shares * position.shift(1).abs() * close.shift(1)
  equity = capital + (daily_pnl - daily_fees).cumsum()
  return equity

In [4]:

#Moving Average

capital = 1000000
def Moving_Average(df , wind_s1 , wind_s2) :
  SMA_fast = df['Close'].rolling(window = wind_s1).mean()
  SMA_slow = df['Close'].rolling(window = wind_s2).mean()

  Signal = (SMA_fast > SMA_slow).astype(int)

  return Signal

#Breakout Strategy

def Breakout_strategy(df , window_sz) :
  wind_high = df['Close'].shift(1).rolling(window = window_sz).max()
  wind_low = df['Close'].shift(1).rolling(window = window_sz).min()

  raw_signal = np.select(
      [df['Close'] > wind_high , df['Close'] < wind_low] ,
      [1,0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal, index=df.index).ffill().fillna(0)

  return signal

#RSI mean reversion Strategy

def RSI_MR(df , wind_sz) :

  delta = df['Close'].diff()

  gain = delta.clip(lower = 0)
  loss = -delta.clip(upper = 0)

  avg_gain = gain.rolling(window = wind_sz).mean()
  avg_loss = loss.rolling(window = wind_sz).mean()

  rs = avg_gain / avg_loss

  rs_val = 100 - (100 / (1 +  rs))

  raw_signal = np.select(
      [(rs_val.shift(1) < 30) & (rs_val >= 30) , (rs_val.shift(1) < 50) & (rs_val >= 50)] ,
      [1 , 0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal , index = df.index).ffill().fillna(0)

  return signal

In [13]:
stocks = {
    "HDFCBANK": df1,
    "TCS": df2,
    "RELIANCE": df3,
    "SUNPHARMA": df4,
    "HINDUNILVR": df5,
    "BHARTIARTL": df6,
    "M&M": df7,
    "LT": df8,
}

results = {}

for stock , df in stocks.items() :
    sigs = {"MA": Moving_Average(df, 10, 20),
            "Breakout": Breakout_strategy(df, 20),
            "RSI": RSI_MR(df, 14)}
    results[stock] = pd.DataFrame(
        {s : backtest(sig , df['Close']) for s , sig in sigs.items()}
    )

net_results = pd.concat(results , axis = 1)
net_results

HDFCBANK                                             TCS  \
                        MA        Breakout             RSI              MA   
Date                                                                         
2020-01-01             NaN             NaN             NaN             NaN   
2020-01-02  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-03  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-06  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-07  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
...                    ...             ...             ...             ...   
2026-11-16   939523.243895   967046.962159   965345.686147   947553.420143   
2026-11-17   939521.693907   967045.412171   965345.686147   947553.420143   
2026-11-18   939514.893920   967038.612184   965345.686147   947553.420143   
2026-11-19   939528.543883   967052.262147   965345.686147   947553.420143   
2026-11-20   939552.743895   967076.462159   965345.686147   947553.420143   

                                                  RELIANCE                  \
                  Breakout             RSI              MA        Breakout   
Date                                                                         
2020-01-01             NaN             NaN             NaN             NaN   
2020-01-02  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-03  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-06  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-07  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
...                    ...             ...             ...             ...   
2026-11-16   967940.026100   963296.785283   942453.993186   969629.554131   
2026-11-17   967941.626198   963296.785283   942476.193137   969651.754082   
2026-11-18   967914.526100   963296.785283   942472.193137   969647.754082   
2026-11-19   967289.907350   963296.785283   942484.793112   969660.354058   
2026-11-20   967289.907350   963296.785283   942486.693137   969662.254082   

                                 SUNPHARMA  ...      HINDUNILVR  \
                       RSI              MA  ...             RSI   
Date                                        ...                   
2020-01-01             NaN             NaN  ...             NaN   
2020-01-02  1000000.000000  1000000.000000  ...  1000000.000000   
2020-01-03  1000000.000000  1000000.000000  ...  1000000.000000   
2020-01-06  1000000.000000  1000000.000000  ...  1000000.000000   
2020-01-07  1000000.000000  1000000.000000  ...  1000000.000000   
...                    ...             ...  ...             ...   
2026-11-16   966311.576089   947807.328696  ...   961222.362262   
2026-11-17   966311.576089   947809.828696  ...   961222.362262   
2026-11-18   966311.576089   947811.328696  ...   961222.362262   
2026-11-19   966311.576089   947839.828696  ...   961222.362262   
2026-11-20   966311.576089   947824.728721  ...   961222.362262   

                BHARTIARTL                                             M&M  \
                        MA        Breakout             RSI              MA   
Date                                                                         
2020-01-01             NaN             NaN             NaN             NaN   
2020-01-02  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-03  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-06  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
2020-01-07  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
...                    ...             ...             ...             ...   
2026-11-16   953300.666677   976998.810883   973952.083023   944219.027673   
2026-11-17   953285.466726   976983.610932   973952.083023   944279.027673   
2026-11-18   953279.166677   976977